In [1]:
pip install pandas numpy geopandas scikit-learn folium pyproj geopy openpyxl shapely scipy


   ---------------------------------------- 0.0/6.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/6.3 MB ? eta -:--:--
   --- ------------------------------------ 0.5/6.3 MB 929.6 kB/s eta 0:00:07
   ------ --------------------------------- 1.0/6.3 MB 1.7 MB/s eta 0:00:04
   ----------- ---------------------------- 1.8/6.3 MB 2.1 MB/s eta 0:00:03
   -------------- ------------------------- 2.4/6.3 MB 2.2 MB/s eta 0:00:02
   ------------------ --------------------- 2.9/6.3 MB 2.4 MB/s eta 0:00:02
   ----------------------- ---------------- 3.7/6.3 MB 2.5 MB/s eta 0:00:02
   ---------------------------- ----------- 4.5/6.3 MB 2.6 MB/s eta 0:00:01
   --------------------------------- ------ 5.2/6.3 MB 2.8 MB/s eta 0:00:01
   -------------------------------------- - 6.0/6.3 MB 2.9 MB/s eta 0:00:01
   ---------------------------------------- 6.3/6.3 MB 2.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------------------ ---------

In [14]:
import pandas as pd
df = pd.read_csv('C:/Users/Administrator/Downloads/HNG_stage3/NIGER_crosschecked.csv')
df.head()
df.columns
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4394 entries, 0 to 4393
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   State                   4394 non-null   object
 1   LGA                     4394 non-null   object
 2   Ward                    4394 non-null   object
 3   PU-Code                 4394 non-null   object
 4   PU-Name                 4394 non-null   object
 5   Accredited_Voters       4394 non-null   int64 
 6   Registered_Voters       4394 non-null   int64 
 7   Results_Found           4394 non-null   bool  
 8   Transcription_Count     4394 non-null   int64 
 9   Result_Sheet_Stamped    4394 non-null   bool  
 10  Result_Sheet_Corrected  4394 non-null   bool  
 11  Result_Sheet_Invalid    4394 non-null   bool  
 12  Result_Sheet_Unclear    4394 non-null   bool  
 13  Result_Sheet_Unsigned   4394 non-null   object
 14  APC                     4394 non-null   int64 
 15  LP  

In [11]:
df = df.rename(columns={'latitude_col_name':'lat','longitude_col_name':'lon','pu_col':'pu_id'})

In [12]:
vote_cols = ['PDP','APC','LP']  # change to actual party column names
df[vote_cols] = df[vote_cols].apply(pd.to_numeric, errors='coerce').fillna(0)


Add coordinates (geocoding)

In [16]:
from geopy.geocoders import Nominatim
from tqdm import tqdm
import time

geolocator = Nominatim(user_agent="inec_outlier_detector")

def geocode_pu(row):
    query = f"{row['PU-Name']}, {row['Ward']}, {row['LGA']}, Niger State, Nigeria"
    try:
        location = geolocator.geocode(query, timeout=10)
        if location:
            return pd.Series([location.latitude, location.longitude])
    except:
        pass
    return pd.Series([None, None])

# Create lat/lon columns
df[['lat','lon']] = None, None

for idx, row in tqdm(df.iterrows(), total=len(df)):
    latlon = geocode_pu(row)
    df.at[idx,'lat'] = latlon[0]
    df.at[idx,'lon'] = latlon[1]
    time.sleep(1)  # prevent rate limit issues


100%|██████████| 4394/4394 [1:58:46<00:00,  1.62s/it]  


In [18]:
missing = df['lat'].isna().sum()
print(f"Missing coordinates after geocoding: {missing}")

# Optionally export to CSV for manual correction
df[df['lat'].isna()].to_csv('missing_coordinates.csv', index=False)
df.head()

Missing coordinates after geocoding: 4392


,State,LGA,Ward,PU-Code,PU-Name,Accredited_Voters,Registered_Voters,Results_Found,Transcription_Count,Result_Sheet_Stamped,...,Result_Sheet_Invalid,Result_Sheet_Unclear,Result_Sheet_Unsigned,APC,LP,PDP,NNPP,Results_File,lat,lon
0,NIGER,AGAIE,BARO,26-01-01-001,AKWANO,164,541,True,-1,False,...,False,False,UNKNOWN,77,0,87,0,https://inec-cvr-cache.s3.eu-west-1.amazonaws....,None,None
1,NIGER,AGAIE,BARO,26-01-01-003,BARO II,148,429,True,-1,False,...,False,False,UNKNOWN,95,0,37,2,https://inec-cvr-cache.s3.eu-west-1.amazonaws....,None,None
2,NIGER,AGAIE,BARO,26-01-01-004,BARO II,148,429,True,-1,False,...,False,False,UNKNOWN,88,0,106,2,https://inec-cvr-cache.s3.eu-west-1.amazonaws....,None,None
3,NIGER,AGAIE,BARO,26-01-01-005,ESSUN,329,803,True,-1,False,...,False,False,UNKNOWN,109,0,206,11,https://inec-cvr-cache.s3.eu-west-1.amazonaws....,None,None
4,NIGER,AGAIE,BARO,26-01-01-006,EVUNTAGI,281,531,True,-1,False,...,False,False,UNKNOWN,175,0,95,3,https://inec-cvr-cache.s3.eu-west-1.amazonaws....,None,None
